%md
Análises - Yellow Taxis NYC (Jan-Mai 2023)


%md
Este notebook responde às perguntas de negócio utilizando os dados da camada Silver (Arquitetura Medallion) no Unity Catalog.

Pré-requisito: Execute o notebook src/ingestion antes deste.

Perguntas:
Qual a média de valor total (total_amount) recebido em um mês considerando todos os yellow táxis da frota?
Qual a média de passageiros (passenger_count) por cada hora do dia que pegaram táxi no mês de maio considerando todos os táxis da frota?


%md

Configuração

In [ ]:
from pyspark.sql import functions as F

# === AJUSTE O CATÁLOGO CONFORME SEU AMBIENTE ===
CATALOG = "main"

SILVER_TABLE = f"{CATALOG}.silver.yellow_taxi_trips"


%md

Carregamento dos Dados Silver


In [ ]:
df_silver = spark.table(SILVER_TABLE)
print(f"Total de registros na Silver: {df_silver.count()}")


%md

Pergunta 1
Qual a média de valor total (total_amount) recebido em um mês considerando todos os yellow táxis da frota?

Resposta via SQL

In [ ]:
display(spark.sql(f"""
    SELECT
        ano_mes,
        ROUND(AVG(total_amount), 2) AS media_total_amount,
        COUNT(*) AS total_corridas
    FROM {SILVER_TABLE}
    GROUP BY ano_mes
    ORDER BY ano_mes
"""))


%md
Resposta via PySpark


In [ ]:
resultado_p1 = (
    df_silver
    .groupBy("ano_mes")
    .agg(
        F.round(F.avg("total_amount"), 2).alias("media_total_amount"),
        F.count("*").alias("total_corridas"),
    )
    .orderBy("ano_mes")
)


In [ ]:
display(resultado_p1)


%md

Pergunta 2
Qual a média de passageiros (passenger_count) por cada hora do dia que pegaram táxi no mês de maio considerando todos os táxis da frota?

Resposta via SQL

In [ ]:
display(spark.sql(f"""
    SELECT
        HOUR(tpep_pickup_datetime) AS hora_do_dia,
        ROUND(AVG(passenger_count), 2) AS media_passageiros,
        COUNT(*) AS total_corridas
    FROM {SILVER_TABLE}
    WHERE ano_mes = '2023-05'
    GROUP BY HOUR(tpep_pickup_datetime)
    ORDER BY hora_do_dia
"""))


%md
Resposta via PySpark


In [ ]:
resultado_p2 = (
    df_silver
    .filter(F.col("ano_mes") == "2023-05")
    .withColumn("hora_do_dia", F.hour(F.col("tpep_pickup_datetime")))
    .groupBy("hora_do_dia")
    .agg(
        F.round(F.avg("passenger_count"), 2).alias("media_passageiros"),
        F.count("*").alias("total_corridas"),
    )
    .orderBy("hora_do_dia")
)


In [ ]:
display(resultado_p2)
